In [1]:
import numpy as np

In [2]:
def encoder_layer(x, params):

    W_Q = params["W_Q"]
    W_K = params["W_K"]
    W_V = params["W_V"]

    Q = x @ W_Q
    K = x @ W_K
    V = x @ W_V

    Q = Q.reshape(len(x), 4, 8)
    K = K.reshape(len(x), 4, 8)
    V = V.reshape(len(x), 4, 8)

    attention_outputs = []
    attention_scores = []

    for h in range(4):

        q = Q[:, h, :]
        k = K[:, h, :]
        v = V[:, h, :]

        scores = (q @ k.T) / np.sqrt(8)

        exp_scores = np.exp(
            scores - np.max(scores, axis=1, keepdims=True)
        )

        scores = exp_scores / exp_scores.sum(
            axis=1, keepdims=True
        )

        attention = scores @ v

        attention_outputs.append(attention)
        attention_scores.append(scores)

    mul_att = np.concatenate(attention_outputs, axis=1)

    residual1 = mul_att + x

    mean1 = residual1.mean(axis=1, keepdims=True)
    variance1 = residual1.var(axis=1, keepdims=True)

    normalized1 = (
        residual1 - mean1
    ) / np.sqrt(variance1 + 1e-5)

    gamma1 = params["gamma1"]
    beta1 = params["beta1"]

    output = gamma1 * normalized1 + beta1

    W1 = params["W1"]
    W2 = params["W2"]
    B1 = params["B1"]
    B2 = params["B2"]

    z1 = output @ W1 + B1
    a1 = np.maximum(0, z1)
    z2 = a1 @ W2 + B2

    residual2 = z2 + output

    mean2 = residual2.mean(axis=1, keepdims=True)
    variance2 = residual2.var(axis=1, keepdims=True)

    normalized2 = (
        residual2 - mean2
    ) / np.sqrt(variance2 + 1e-5)

    gamma2 = params["gamma2"]
    beta2 = params["beta2"]

    encoder_output = gamma2 * normalized2 + beta2

    cache = {
        "input": x,

        "Q": Q,
        "K": K,
        "V": V,
        "attention_scores": attention_scores,
        "attention_outputs": attention_outputs,
        "mul_att": mul_att,

        "residual1": residual1,
        "mean1": mean1,
        "variance1": variance1,
        "normalized1": normalized1,
        "output": output,

        "z1": z1,
        "a1": a1,
        "z2": z2,

        "residual2": residual2,
        "mean2": mean2,
        "variance2": variance2,
        "normalized2": normalized2,

        "encoder_output": encoder_output
    }

    return encoder_output, cache

In [11]:
def create_encoder_params():

    return {
        "W_Q": np.random.randn(32, 32) * 0.02,
        "W_K": np.random.randn(32, 32) * 0.02,
        "W_V": np.random.randn(32, 32) * 0.02,

        "W1": np.random.randn(32, 128) * 0.02,
        "W2": np.random.randn(128, 32) * 0.02,

        "B1": np.zeros(128),
        "B2": np.zeros(32),

        "gamma1": np.ones(32),
        "beta1": np.zeros(32),

        "gamma2": np.ones(32),
        "beta2": np.zeros(32)
    }


def create_decoder_params():

    return {
        # Self attention
        "W_Q_self": np.random.randn(32, 32) * 0.02,
        "W_K_self": np.random.randn(32, 32) * 0.02,
        "W_V_self": np.random.randn(32, 32) * 0.02,

        # Cross attention
        "W_Q_cross": np.random.randn(32, 32) * 0.02,
        "W_K_cross": np.random.randn(32, 32) * 0.02,
        "W_V_cross": np.random.randn(32, 32) * 0.02,

        # FFN
        "W1": np.random.randn(32, 128) * 0.02,
        "W2": np.random.randn(128, 32) * 0.02,

        "B1": np.zeros(128),
        "B2": np.zeros(32),

        # LayerNorm 1
        "gamma1": np.ones(32),
        "beta1": np.zeros(32),

        # LayerNorm 2
        "gamma2": np.ones(32),
        "beta2": np.zeros(32),

        # LayerNorm 3
        "gamma3": np.ones(32),
        "beta3": np.zeros(32)
    }


# =========================================================
# 6 ENCODER LAYERS
# =========================================================

encoder_params = []

for i in range(6):
    encoder_params.append(
        create_encoder_params()
    )
decoder_params = []

for i in range(6):
    decoder_params.append(
        create_decoder_params()
    )

W_out = (
    np.random.randn(
        32,
        len(vocab_output)
    ) * 0.02
)

b_out = np.zeros(
    len(vocab_output)
)


print("Encoder layers:", len(encoder_params))
print("Decoder layers:", len(decoder_params))
print("W_out shape:", W_out.shape)

Encoder layers: 6
Decoder layers: 6
W_out shape: (32, 7)


In [12]:
def decoder_layer(x, encoder_output, params):
    W_Q_self = params["W_Q_self"]
    W_K_self = params["W_K_self"]
    W_V_self = params["W_V_self"]

    Q_self = x @ W_Q_self
    K_self = x @ W_K_self
    V_self = x @ W_V_self

    Q_self = Q_self.reshape(len(x), 4, 8)
    K_self = K_self.reshape(len(x), 4, 8)
    V_self = V_self.reshape(len(x), 4, 8)

    self_attention_outputs = []
    self_attention_scores = []

    for h in range(4):

        q = Q_self[:, h, :]
        k = K_self[:, h, :]
        v = V_self[:, h, :]

        scores = (q @ k.T) / np.sqrt(8)

        # causal mask
        for i in range(len(scores)):
            for j in range(i + 1, len(scores)):
                scores[i, j] = -np.inf

        exp_scores = np.exp(
            scores - np.max(scores, axis=1, keepdims=True)
        )

        scores = exp_scores / exp_scores.sum(
            axis=1, keepdims=True
        )

        attention = scores @ v

        self_attention_outputs.append(attention)
        self_attention_scores.append(scores)

    self_att = np.concatenate(
        self_attention_outputs,
        axis=1
    )

    residual1 = self_att + x

    mean1 = residual1.mean(axis=1, keepdims=True)
    variance1 = residual1.var(axis=1, keepdims=True)

    normalized1 = (
        residual1 - mean1
    ) / np.sqrt(variance1 + 1e-5)

    gamma1 = params["gamma1"]
    beta1 = params["beta1"]

    output1 = gamma1 * normalized1 + beta1

    # CROSS ATTENTION

    W_Q_cross = params["W_Q_cross"]
    W_K_cross = params["W_K_cross"]
    W_V_cross = params["W_V_cross"]

    Q_cross = output1 @ W_Q_cross
    K_cross = encoder_output @ W_K_cross
    V_cross = encoder_output @ W_V_cross

    Q_cross = Q_cross.reshape(len(output1), 4, 8)
    K_cross = K_cross.reshape(len(encoder_output), 4, 8)
    V_cross = V_cross.reshape(len(encoder_output), 4, 8)

    cross_attention_outputs = []
    cross_attention_scores = []

    for h in range(4):

        q = Q_cross[:, h, :]
        k = K_cross[:, h, :]
        v = V_cross[:, h, :]

        scores = (q @ k.T) / np.sqrt(8)

        exp_scores = np.exp(
            scores - np.max(scores, axis=1, keepdims=True)
        )

        scores = exp_scores / exp_scores.sum(
            axis=1, keepdims=True
        )

        attention = scores @ v

        cross_attention_outputs.append(attention)
        cross_attention_scores.append(scores)

    cross_att = np.concatenate(
        cross_attention_outputs,
        axis=1
    )

    residual2 = cross_att + output1

    mean2 = residual2.mean(axis=1, keepdims=True)
    variance2 = residual2.var(axis=1, keepdims=True)

    normalized2 = (
        residual2 - mean2
    ) / np.sqrt(variance2 + 1e-5)

    gamma2 = params["gamma2"]
    beta2 = params["beta2"]

    output2 = gamma2 * normalized2 + beta2
    # FFN
    W1 = params["W1"]
    W2 = params["W2"]
    B1 = params["B1"]
    B2 = params["B2"]

    z1 = output2 @ W1 + B1
    a1 = np.maximum(0, z1)
    z2 = a1 @ W2 + B2

    residual3 = z2 + output2

    mean3 = residual3.mean(axis=1, keepdims=True)
    variance3 = residual3.var(axis=1, keepdims=True)

    normalized3 = (
        residual3 - mean3
    ) / np.sqrt(variance3 + 1e-5)

    gamma3 = params["gamma3"]
    beta3 = params["beta3"]

    decoder_output = gamma3 * normalized3 + beta3
    # CACHE
    cache = {

        "input": x,

        # self attention
        "Q_self": Q_self,
        "K_self": K_self,
        "V_self": V_self,
        "self_attention_scores": self_attention_scores,
        "self_attention_outputs": self_attention_outputs,
        "self_att": self_att,

        "residual1": residual1,
        "mean1": mean1,
        "variance1": variance1,
        "normalized1": normalized1,
        "output1": output1,

        # cross attention
        "Q_cross": Q_cross,
        "K_cross": K_cross,
        "V_cross": V_cross,
        "cross_attention_scores": cross_attention_scores,
        "cross_attention_outputs": cross_attention_outputs,
        "cross_att": cross_att,

        "residual2": residual2,
        "mean2": mean2,
        "variance2": variance2,
        "normalized2": normalized2,
        "output2": output2,

        # FFN
        "z1": z1,
        "a1": a1,
        "z2": z2,

        "residual3": residual3,
        "mean3": mean3,
        "variance3": variance3,
        "normalized3": normalized3,

        "decoder_output": decoder_output
    }

    return decoder_output, cache

In [13]:
def attention_backward(
    d_output,
    Q,
    K,
    V,
    attention_scores,
    causal=False
):

    d_output_heads = d_output.reshape(
        len(d_output), 4, 8
    )

    dQ_heads = []
    dK_heads = []
    dV_heads = []

    for h in range(4):

        q = Q[:, h, :]
        k = K[:, h, :]
        v = V[:, h, :]

        A = attention_scores[h]

        dO = d_output_heads[:, h, :]

        # O = A @ V
        dA = dO @ v.T
        dV = A.T @ dO

        # softmax backward
        dS = A * (
            dA -
            (dA * A).sum(
                axis=1,
                keepdims=True
            )
        )

        # causal mask
        if causal:
            for i in range(len(dS)):
                for j in range(i + 1, len(dS)):
                    dS[i, j] = 0

        # S = Q @ K.T / sqrt(d_k)
        dS = dS / np.sqrt(8)

        dQ = dS @ k
        dK = dS.T @ q

        dQ_heads.append(dQ)
        dK_heads.append(dK)
        dV_heads.append(dV)

    dQ = np.stack(dQ_heads, axis=1)
    dK = np.stack(dK_heads, axis=1)
    dV = np.stack(dV_heads, axis=1)

    return dQ, dK, dV

In [14]:
def layer_norm_backward(
    d_output,
    normalized,
    variance,
    gamma
):

    d_normalized = d_output * gamma

    d_input = (
        d_normalized
        - d_normalized.mean(
            axis=1,
            keepdims=True
        )
        - normalized * (
            d_normalized * normalized
        ).mean(
            axis=1,
            keepdims=True
        )
    ) / np.sqrt(variance + 1e-5)

    d_gamma = np.sum(
        d_output * normalized,
        axis=0
    )

    d_beta = np.sum(
        d_output,
        axis=0
    )

    return d_input, d_gamma, d_beta

In [15]:
def ffn_backward(dz2, output2, z1, a1, W1, W2):

    dW2 = a1.T @ dz2
    dB2 = np.sum(dz2, axis=0)

    da1 = dz2 @ W2.T
    dz1 = da1 * (z1 > 0)

    dW1 = output2.T @ dz1
    dB1 = np.sum(dz1, axis=0)

    d_output2 = dz1 @ W1.T

    return d_output2, dW1, dB1, dW2, dB2

In [16]:
input_sentence = "I am a boy"

s = input_sentence.split()

vocab = {
    "<pad>": 0,
    "<sos>": 1,
    "<eos>": 2
}

for i in range(len(s)):
    vocab[s[i]] = i + 3

token_id = np.array(
    [vocab["<sos>"]]
    + [vocab[token] for token in s]
    + [vocab["<eos>"]]
)

# Encoder embedding
encoder_embedding_matrix = (
    np.random.randn(len(vocab), 32) * 0.02
)

embedding = encoder_embedding_matrix[token_id]

# positional embedding
pos_emb = np.random.randn(len(token_id), 32) * 0.02

embedding = embedding + pos_emb

output_sentence = "Main ek ladka hoon"

s = output_sentence.split()

vocab_output = {
    "<pad>": 0,
    "<sos>": 1,
    "<eos>": 2
}

for i in range(len(s)):
    vocab_output[s[i]] = i + 3


# Full target sequence
full_output_ids = np.array(
    [vocab_output["<sos>"]]
    + [vocab_output[token] for token in s]
    + [vocab_output["<eos>"]]
)

print("Full output IDs:", full_output_ids)

decoder_input_ids = full_output_ids[:-1]

target_ids = full_output_ids[1:]


print("Decoder input IDs:", decoder_input_ids)
print("Target IDs:", target_ids)

decoder_embedding_matrix = (
    np.random.randn(len(vocab_output), 32) * 0.02
)

embedding_output = decoder_embedding_matrix[
    decoder_input_ids
]

# positional embedding
output_pos_emb = (
    np.random.randn(len(decoder_input_ids), 32) * 0.02
)

embedding_output = (
    embedding_output + output_pos_emb
)

Full output IDs: [1 3 4 5 6 2]
Decoder input IDs: [1 3 4 5 6]
Target IDs: [3 4 5 6 2]


In [18]:
W_out = np.random.randn(
    32,
    len(vocab_output)
) * 0.01

b_out = np.zeros(
    len(vocab_output)
)

lr = 0.001
epochs = 1000

for epoch in range(epochs):
    encoder = embedding
    encoder_cache = []

    for i in range(6):

        encoder, cache = encoder_layer(
            encoder,
            encoder_params[i]
        )

        encoder_cache.append(cache)

    encoder6 = encoder
    decoder = embedding_output
    decoder_cache = []

    for i in range(6):

        decoder, cache = decoder_layer(
            decoder,
            encoder6,
            decoder_params[i]
        )

        decoder_cache.append(cache)

    decoder6 = decoder

    logits = decoder6 @ W_out + b_out

    exp_logits = np.exp(
        logits -
        np.max(
            logits,
            axis=1,
            keepdims=True
        )
    )

    probs = exp_logits / exp_logits.sum(
        axis=1,
        keepdims=True
    )
    loss = 0

    for i in range(len(target_ids)):

        loss += -np.log(
            probs[i, target_ids[i]] + 1e-9
        )

    loss /= len(target_ids)

    target_one_hot = np.zeros_like(probs)

    for i, token_id in enumerate(target_ids):

        target_one_hot[i, token_id] = 1

    d_logits = (
        probs - target_one_hot
    ) / len(target_ids)


    dW_out = decoder6.T @ d_logits
    db_out = d_logits.sum(axis=0)

    d_decoder = d_logits @ W_out.T

    d_encoder_total = np.zeros_like(encoder6)

    for i in range(5, -1, -1):

        params = decoder_params[i]
        cache = decoder_cache[i]


        d_residual3, d_gamma3, d_beta3 = \
            layer_norm_backward(
                d_decoder,
                cache["normalized3"],
                cache["variance3"],
                params["gamma3"]
            )


        # residual3 = z2 + output2

        dz2 = d_residual3

        d_output2 = d_residual3

        dW2 = cache["a1"].T @ dz2
        dB2 = dz2.sum(axis=0)

        da1 = dz2 @ params["W2"].T

        dz1 = da1 * (
            cache["z1"] > 0
        )

        dW1 = cache["output2"].T @ dz1
        dB1 = dz1.sum(axis=0)

        d_output2 += dz1 @ params["W1"].T

        d_residual2, d_gamma2, d_beta2 = \
            layer_norm_backward(
                d_output2,
                cache["normalized2"],
                cache["variance2"],
                params["gamma2"]
            )


        # residual2 = cross_att + output1

        d_cross_att = d_residual2
        d_output1 = d_residual2

        dQ_cross, dK_cross, dV_cross = \
            attention_backward(
                d_cross_att,
                cache["Q_cross"],
                cache["K_cross"],
                cache["V_cross"],
                cache["cross_attention_scores"],
                causal=False
            )

        dQ_cross_2d = dQ_cross.reshape(
            len(dQ_cross), 32
        )

        dK_cross_2d = dK_cross.reshape(
            len(dK_cross), 32
        )

        dV_cross_2d = dV_cross.reshape(
            len(dV_cross), 32
        )

        dW_Q_cross = (
            cache["output1"].T @
            dQ_cross_2d
        )

        dW_K_cross = (
            encoder6.T @
            dK_cross_2d
        )

        dW_V_cross = (
            encoder6.T @
            dV_cross_2d
        )

        d_output1 += (
            dQ_cross_2d @
            params["W_Q_cross"].T
        )


        # Gradient flowing into encoder6
        d_encoder_total += (
            dK_cross_2d @
            params["W_K_cross"].T
            +
            dV_cross_2d @
            params["W_V_cross"].T
        )

        # LayerNorm 1
        d_residual1, d_gamma1, d_beta1 = \
            layer_norm_backward(
                d_output1,
                cache["normalized1"],
                cache["variance1"],
                params["gamma1"]
            )


        # residual1 = self_att + input

        d_self_att = d_residual1

        d_decoder_input = d_residual1
        # SELF ATTENTION
        dQ_self, dK_self, dV_self = \
            attention_backward(
                d_self_att,
                cache["Q_self"],
                cache["K_self"],
                cache["V_self"],
                cache["self_attention_scores"],
                causal=True
            )
        # Self-attention projections

        dQ_self_2d = dQ_self.reshape(
            len(dQ_self), 32
        )

        dK_self_2d = dK_self.reshape(
            len(dK_self), 32
        )

        dV_self_2d = dV_self.reshape(
            len(dV_self), 32
        )

        dW_Q_self = (
            cache["input"].T @
            dQ_self_2d
        )

        dW_K_self = (
            cache["input"].T @
            dK_self_2d
        )

        dW_V_self = (
            cache["input"].T @
            dV_self_2d
        )


        d_decoder_input += (
            dQ_self_2d @
            params["W_Q_self"].T
            +
            dK_self_2d @
            params["W_K_self"].T
            +
            dV_self_2d @
            params["W_V_self"].T
        )

        d_decoder = d_decoder_input



        # SAVE GRADIENTS

        params["dW_Q_self"] = dW_Q_self
        params["dW_K_self"] = dW_K_self
        params["dW_V_self"] = dW_V_self

        params["dW_Q_cross"] = dW_Q_cross
        params["dW_K_cross"] = dW_K_cross
        params["dW_V_cross"] = dW_V_cross

        params["dW1"] = dW1
        params["dW2"] = dW2

        params["dB1"] = dB1
        params["dB2"] = dB2

        params["dgamma1"] = d_gamma1
        params["dbeta1"] = d_beta1

        params["dgamma2"] = d_gamma2
        params["dbeta2"] = d_beta2

        params["dgamma3"] = d_gamma3
        params["dbeta3"] = d_beta3

    d_encoder = d_encoder_total

    for i in range(5, -1, -1):

        params = encoder_params[i]
        cache = encoder_cache[i]

        # LayerNorm 2
        d_residual2, d_gamma2, d_beta2 = \
            layer_norm_backward(
                d_encoder,
                cache["normalized2"],
                cache["variance2"],
                params["gamma2"]
            )


        # residual2 = z2 + output

        dz2 = d_residual2
        d_output = d_residual2

        dW2 = cache["a1"].T @ dz2
        dB2 = dz2.sum(axis=0)

        da1 = dz2 @ params["W2"].T

        dz1 = da1 * (
            cache["z1"] > 0
        )

        dW1 = cache["output"].T @ dz1
        dB1 = dz1.sum(axis=0)

        d_output += dz1 @ params["W1"].T


        d_residual1, d_gamma1, d_beta1 = \
            layer_norm_backward(
                d_output,
                cache["normalized1"],
                cache["variance1"],
                params["gamma1"]
            )


        # residual1 = mul_att + input

        d_mul_att = d_residual1

        d_encoder_input = d_residual1


        dQ, dK, dV = attention_backward(
            d_mul_att,
            cache["Q"],
            cache["K"],
            cache["V"],
            cache["attention_scores"],
            causal=False
        )


        dQ_2d = dQ.reshape(
            len(dQ), 32
        )

        dK_2d = dK.reshape(
            len(dK), 32
        )

        dV_2d = dV.reshape(
            len(dV), 32
        )
        dW_Q = (
            cache["input"].T @
            dQ_2d
        )

        dW_K = (
            cache["input"].T @
            dK_2d
        )

        dW_V = (
            cache["input"].T @
            dV_2d
        )


        d_encoder_input += (
            dQ_2d @ params["W_Q"].T
            +
            dK_2d @ params["W_K"].T
            +
            dV_2d @ params["W_V"].T
        )
        d_encoder = d_encoder_input

        params["dW_Q"] = dW_Q
        params["dW_K"] = dW_K
        params["dW_V"] = dW_V

        params["dW1"] = dW1
        params["dW2"] = dW2

        params["dB1"] = dB1
        params["dB2"] = dB2

        params["dgamma1"] = d_gamma1
        params["dbeta1"] = d_beta1

        params["dgamma2"] = d_gamma2
        params["dbeta2"] = d_beta2

    for params in encoder_params:

        for name in [
            "W_Q",
            "W_K",
            "W_V",
            "W1",
            "W2",
            "B1",
            "B2",
            "gamma1",
            "beta1",
            "gamma2",
            "beta2"
        ]:

            params[name] -= lr * params["d" + name]

    for params in decoder_params:

        for name in [
            "W_Q_self",
            "W_K_self",
            "W_V_self",
            "W_Q_cross",
            "W_K_cross",
            "W_V_cross",
            "W1",
            "W2",
            "B1",
            "B2",
            "gamma1",
            "beta1",
            "gamma2",
            "beta2",
            "gamma3",
            "beta3"
        ]:

            params[name] -= lr * params["d" + name]
    # Output layer

    W_out -= lr * dW_out
    b_out -= lr * db_out
    if epoch % 10 == 0:

        predicted_ids = np.argmax(
            probs,
            axis=1
        )

        id_to_word = {
            v: k
            for k, v in vocab_output.items()
        }

        predicted_words = [
            id_to_word[i]
            for i in predicted_ids
        ]

        print(
            f"Epoch {epoch:4d} | "
            f"Loss: {loss:.6f} | "
            f"Prediction: {predicted_words}"
        )

Epoch    0 | Loss: 1.950429 | Prediction: ['ladka', 'Main', 'ek', '<pad>', 'ladka']
Epoch   10 | Loss: 1.883640 | Prediction: ['Main', 'ek', 'ek', '<pad>', 'ladka']
Epoch   20 | Loss: 1.818599 | Prediction: ['Main', 'ek', 'ek', 'hoon', '<eos>']
Epoch   30 | Loss: 1.755320 | Prediction: ['Main', 'ek', 'ladka', 'hoon', '<eos>']
Epoch   40 | Loss: 1.693814 | Prediction: ['Main', 'ek', 'ladka', 'hoon', '<eos>']
Epoch   50 | Loss: 1.634089 | Prediction: ['Main', 'ek', 'ladka', 'hoon', '<eos>']
Epoch   60 | Loss: 1.576150 | Prediction: ['Main', 'ek', 'ladka', 'hoon', '<eos>']
Epoch   70 | Loss: 1.520000 | Prediction: ['Main', 'ek', 'ladka', 'hoon', '<eos>']
Epoch   80 | Loss: 1.465637 | Prediction: ['Main', 'ek', 'ladka', 'hoon', '<eos>']
Epoch   90 | Loss: 1.413057 | Prediction: ['Main', 'ek', 'ladka', 'hoon', '<eos>']
Epoch  100 | Loss: 1.362249 | Prediction: ['Main', 'ek', 'ladka', 'hoon', '<eos>']
Epoch  110 | Loss: 1.313201 | Prediction: ['Main', 'ek', 'ladka', 'hoon', '<eos>']
Epoch  1